# Batched greedy decoding with EOS in lax.scan

A compiled decode loop has a fixed number of steps and fixed shapes, so sequences that finish early keep running but emit padding. Write that loop with `jax.lax.scan` for a toy recurrent language model.

Model, with `params = {'emb': (V, D), 'w': (D, D), 'out': (D, V)}` and a hidden state $h$ of shape `(D,)` per sequence, starting at zeros:

- Consume a token $t$: $h \leftarrow \tanh(\text{emb}[t] + h\, W)$.
- Predict: logits $= h\, W_\text{out}$; the greedy token is the `argmax` (ties go to the lower id).

Write `greedy_decode(params, prompt, n_steps, eos_id, pad_id)`:

1. `prompt`: `(B, P)` int, `P >= 1`. Consume the prompt tokens in order, for every sequence (one scan).
2. Then run `n_steps` (a Python int) generation steps (a second scan). At each step, predict a token from the current state. If the sequence has **already** emitted `eos_id`, the step's token is `pad_id` instead. Record the token, mark the sequence done if the token is `eos_id`, and consume the token to update $h$.
3. Return `(tokens, lengths)`: `tokens` `(B, n_steps)` int32; `lengths` `(B,)` int32, the number of non-padding tokens generated per sequence, **including** the EOS token.

`0 <= eos_id, pad_id < V`. Process the whole batch at once (batched ops or `vmap`), no Python loops.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def greedy_decode(params, prompt, n_steps, eos_id, pad_id):
    """prompt (B, P). Return (tokens (B, n_steps) int32, lengths (B,) int32)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["emb", {"__nd__": [[3.0, 0.0, 0.0], [0.0, 3.0, 0.0], [0.0, 0.0, 3.0]], "dtype": "float64"}], ["w", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}], ["out", {"__nd__": [[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]], "dtype": "float64"}]]}, {"__nd__": [[0]], "dtype": "int64"}, 4, 2, 0], "e": {"__tup__": [{"__nd__": [[1, 2, 0, 0]], "dtype": "int32"}, {"__nd__": [2], "dtype": "int32"}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[3.0, 0.0, 0.0], [0.0, 3.0, 0.0], [0.0, 0.0, 3.0]], "dtype": "float64"}], ["w", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}], ["out", {"__nd__": [[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]], "dtype": "float64"}]]}, {"__nd__": [[1], [0]], "dtype": "int64"}, 3, 0, 1], "e": {"__tup__": [{"__nd__": [[2, 0, 1], [1, 2, 0]], "dtype": "int32"}, {"__nd__": [2, 3], "dtype": "int32"}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[-0.5878047628908267, 0.07488060228501801, -0.24778524852658915, -0.2519540852256913], [0.09392381626763814, -0.33856662199064713, 0.44538415888733074, -0.07333929020365215], [0.8521045273640873, -0.9855482268709064, -0.5409899837027393, -1.2841245702221362], [-2.504941973078878, -3.3487490695773623, 0.09017949859625311, -1.245763381342525], [-1.1327055361987495, -0.8710711713664405, 0.6265854243281873, 0.8517128554401014], [1.0290255389467757, 1.3871948027255638, 4.473781694646655, 0.44207860475295335]], "dtype": "float64"}], ["w", {"__nd__": [[0.07718359478880035, -1.091100792706506, -0.8420303178204961, 1.0964754416979774], [1.3728931717884494, 0.22317273858985157, -0.04014094899946342, 0.2785444248448195], [-1.0176398815751115, -0.26530357642520896, -0.2027093604885496, 0.8677496405778949], [-0.4956201737272512, 0.1815103938173096, 0.3620511664313069, 0.5401576750320497]], "dtype": "float64"}], ["out", {"__nd__": [[-0.6435444601070928, -1.0306041554019147, -3.2651310910470377, 1.7213813823795494, 1.7733000127979577, -0.7348976193897494], [0.3466989211100454, -2.0940905427667165, -0.3139010841649385, 1.0913548568530995, -0.7251012115161071, 0.0946303766677125], [1.6499637126029483, 0.9006528451233677, -0.9192467923649807, 2.4221076752733652, 2.6619073568981304, 1.505216116397969], [0.901003625744283, 1.073652543018531, 1.322774548307331, -1.8940083120148246, 1.619308719883719, -1.2607168326619898]], "dtype": "float64"}]]}, {"__nd__": [[4, 3, 5, 0], [5, 5, 4, 0], [0, 1, 3, 4]], "dtype": "int64"}, 6, 5, 0], "e": {"__tup__": [{"__nd__": [[3, 1, 5, 0, 0, 0], [5, 0, 0, 0, 0, 0], [5, 0, 0, 0, 0, 0]], "dtype": "int32"}, {"__nd__": [3, 1, 1], "dtype": "int32"}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[-0.5878047628908267, 0.07488060228501801, -0.24778524852658915, -0.2519540852256913], [0.09392381626763814, -0.33856662199064713, 0.44538415888733074, -0.07333929020365215], [0.8521045273640873, -0.9855482268709064, -0.5409899837027393, -1.2841245702221362], [-2.504941973078878, -3.3487490695773623, 0.09017949859625311, -1.245763381342525], [-1.1327055361987495, -0.8710711713664405, 0.6265854243281873, 0.8517128554401014], [1.0290255389467757, 1.3871948027255638, 4.473781694646655, 0.44207860475295335]], "dtype": "float64"}], ["w", {"__nd__": [[0.07718359478880035, -1.091100792706506, -0.8420303178204961, 1.0964754416979774], [1.3728931717884494, 0.22317273858985157, -0.04014094899946342, 0.2785444248448195], [-1.0176398815751115, -0.26530357642520896, -0.2027093604885496, 0.8677496405778949], [-0.4956201737272512, 0.1815103938173096, 0.3620511664313069, 0.5401576750320497]], "dtype": "float64"}], ["out", {"__nd__": [[-0.6435444601070928, -1.0306041554019147, -3.2651310910470377, 1.7213813823795494, 1.7733000127979577, -0.7348976193897494], [0.3466989211100454, -2.0940905427667165, -0.3139010841649385, 1.0913548568530995, -0.7251012115161071, 0.0946303766677125], [1.6499637126029483, 0.9006528451233677, -0.9192467923649807, 2.4221076752733652, 2.6619073568981304, 1.505216116397969], [0.901003625744283, 1.073652543018531, 1.322774548307331, -1.8940083120148246, 1.619308719883719, -1.2607168326619898]], "dtype": "float64"}]]}, {"__nd__": [[1], [1], [0], [0]], "dtype": "int64"}, 8, 0, 1], "e": {"__tup__": [{"__nd__": [[4, 2, 5, 3, 2, 3, 2, 3], [4, 2, 5, 3, 2, 3, 2, 3], [2, 3, 2, 3, 2, 3, 2, 3], [2, 3, 2, 3, 2, 3, 2, 3]], "dtype": "int32"}, {"__nd__": [8, 8, 8, 8], "dtype": "int32"}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[-2.480639982553299, -2.716720474402706, -0.8000385048441679, 2.9863795482865116, -0.42149402436422984, 0.9831558595874119, 1.9160395203535403, -1.9954900378224045], [2.0217372774589095, 1.2764766621710268, 0.801966764085516, 1.4381097219354568, -3.1067875874293387, 0.2401263519321271, -2.3448821720985533, 0.4942320987520903], [-0.2857912265310802, -0.6715318913726713, 0.851373627571516, -2.683356599656093, 0.10000258693416532, 3.260020836390935, 1.5415881752747613, -1.4510071382778886], [0.6223021821955041, -0.2702836267029708, 0.7401917416361372, -1.7876096051130728, 0.706666730424702, 0.0009656921988085963, -1.7556907221493394, 1.047918091185245], [2.056660489546415, 1.7305544852009456, -3.9608966092885414, 0.5222009698294204, 0.5479138294047176, -0.2912238305816843, -0.26268725133788995, 0.6405505283912796], [1.0112646224965198, -1.0893133392892733, 0.3281446313603118, -0.35487844215377956, -1.1263217148950264, 0.5588690160108815, -0.8632570067491525, -2.6483181691610853], [-0.8076400912434525, -2.6660209635922274, 1.3747044946527045, -0.42699882744375217, 1.499047640580237, -3.0524832990509765, 2.4240706165143315, 0.7207565431892479], [-1.720396976934998, 3.0828778299599464, -1.0801764172830997, -1.728117366975378, -0.24436413642127341, 0.6066082964135224, 1.3053310246618215, 3.5165664471439904], [-1.2954186622790398, -2.721531697925885, 0.4649135882383171, -1.9256331415372043, -2.0221648362220757, -0.08958622252086722, 1.994067380696031, -1.2915681374650367], [-1.125862268981024, -0.35386026431267825, -0.5345040864352195, 1.6673602181750873, 0.01696092968568933, -1.9047142083419915, 3.5963176017090266, -0.9587876942470179]], "dtype": "float64"}], ["w", {"__nd__": [[-0.5905746218385005, 0.2086845664166414, 0.014954603133332297, 0.22120453309955948, -0.031348031162297584, 0.007515187475248631, 0.059418816795669724, 0.31906556857123985], [-0.7863176048714314, 0.04591053939590042, 0.0020570598430873005, 0.7808156444411869, -0.2624187984466079, 0.4421371278231991, 0.33546283656332126, -0.2423055388225602], [0.2147668391119947, 0.31733065007877936, 0.45957861939290384, 0.32803069584745975, -0.018074468097302537, 0.19950816232261265, -0.2169838097080372, -0.3174141570843084], [-0.15603350969634014, -0.24766088088276797, -0.2906617987945646, 0.5908626477689349, -0.627856820741104, 0.13742462177717474, 0.14985976551418423, -0.4073285140454484], [0.4923621934393152, -0.3465042237274912, 0.079362608874512, -0.5999157387366058, -0.48967447092575445, -0.29066486147687953, 0.10688127848235708, 0.5562089624169323], [-0.4280680795520406, 0.3026961935629633, -0.2033411967497912, 0.2651156919262848, 0.25917967250452595, 0.07584489970240159, 0.11917724750151544, -0.6611198835288923], [0.25484732566223256, 0.44414232177642204, -0.03583989792735848, 0.2987152733695875, 0.2815725450278133, 0.4439692545256791, -0.2745762017349393, -0.18276754850665725], [0.286079655124401, 0.27384360472395125, 0.5088520082036565, -0.08257369547157234, 0.010328964873013763, -0.24887471247697104, -0.07342449940920209, 0.5017411130967115]], "dtype": "float64"}], ["out", {"__nd__": [[0.030672683579352333, 0.8360786824619768, -0.06098230198675081, -0.4391767752495511, -0.0035918776550691246, -0.03529034476562611, -1.4782947351441915, -1.1260354388134146, -0.28556802561480277, 0.8933924858726634], [-0.1188810709616118, -2.363025005155766, 4.027789870545235, 0.47317321906501436, -0.4719720265128612, 0.17531098885032098, 0.005574167254955952, 0.063443504620669, 0.88204084637612, 0.7268245135807387], [0.26007016235906844, 1.6938288536077943, -1.390108465199785, 3.1511475529931046, 0.43405989964506286, -1.462028492113716, -2.9158478206622123, 0.4528341076768687, 1.1791943120229726, -0.4899806955726518], [-0.305752327650582, -0.5761232824041398, 0.969729955325781, 0.4747209336738202, -2.1802741242557326, 0.6896926043565623, 0.13529697270962673, 0.7483212766572238, -1.2529986399278972, 1.3075792742597008], [2.1521197582417253, -3.1946656783495935, 1.49294058917147, -0.026901756977689092, -0.07108373096617952, -0.35273068703841204, -0.506855834060592, 0.4043082941541547, -0.5207646355163775, -3.6987953224828227], [1.0479578134266774, 0.260554541658924, 1.0208170959368426, -2.736532822853887, 0.7578392090928916, -1.0301252201723146, 1.2637454269058792, 0.4600153770544074, 2.260010564863062, 1.7692579500576193], [1.6650642878619824, -3.269950726153816, 1.9786775589822332, 1.0448024536763845, -0.052737620416471044, 1.9128324752708896, 0.39843382257081406, 0.6632309192653263, 0.8071241259151031, -0.7187389526703543], [0.4710948552344938, -2.083813964368316, 1.2927551956133256, -0.068078119939814, 0.7579056419049621, -1.3061533333607263, -1.0598730430308962, 0.49344882425893566, 2.232618601731236, -2.2057584392765195]], "dtype": "float64"}]]}, {"__nd__": [[4, 3, 4], [4, 2, 1], [9, 1, 7], [3, 5, 2], [5, 8, 3]], "dtype": "int64"}, 10, 3, 9], "e": {"__tup__": [{"__nd__": [[2, 8, 1, 9, 5, 1, 9, 5, 1, 9], [9, 5, 1, 9, 5, 1, 9, 5, 1, 9], [2, 0, 6, 3, 9, 9, 9, 9, 9, 9], [0, 6, 3, 9, 9, 9, 9, 9, 9, 9], [4, 2, 8, 1, 9, 5, 1, 9, 5, 1]], "dtype": "int32"}, {"__nd__": [10, 10, 4, 3, 10], "dtype": "int32"}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[0.6622203153721444, -1.5635845861065256, 0.344390878595728], [0.12457381990847732, -0.21925762283222688, 0.8638185334764528], [-2.0950000278870813, -1.254366368853876, 0.778786692728198], [0.30301449747279857, 0.8108672443246765, 1.2135868884029188]], "dtype": "float64"}], ["w", {"__nd__": [[1.1443001825085775, 0.48769098777385655, 0.8081255009816727], [-0.0021293427067820463, 0.5248070582512276, 0.21113518910218362], [-0.31067717964044556, -0.08831235652205142, -0.18303727306740625]], "dtype": "float64"}], ["out", {"__nd__": [[-0.9177254623168918, -0.7995045436707459, 2.9612296500826516, -1.3989757410019592], [-0.22150637998553518, -3.5407532808468622, -1.0926632114287456, 0.298332349530911], [1.5169939939902266, -1.9959555201047336, 0.0933077742847969, -0.8958745612648005]], "dtype": "float64"}]]}, {"__nd__": [[2, 2], [0, 1]], "dtype": "int64"}, 1, 1, 0], "e": {"__tup__": [{"__nd__": [[1], [2]], "dtype": "int32"}, {"__nd__": [1, 1], "dtype": "int32"}]}}, {"a": [{"__dict__": [["emb", {"__nd__": [[-0.027137073255577453, 0.4529902486144328, -0.05094467109380915, 2.4984829876123715, 0.9113720908024615, 0.9419170711237714], [-2.8789333090184672, 1.7658048757234297, 0.9078859046292622, 2.2527213703310376, -0.7437432894390535, 0.7077514040669078], [3.6649772865783357, 0.3936392673976178, 0.367680562632666, 0.84255722809024, 1.9005111865321391, -0.7113013904472658], [0.4470964869968267, -1.9856746942912822, -2.6311964535599226, -1.2572382868134864, -1.3178180857232604, -2.5952574208547574], [-2.0323264943757082, -1.4779321891164463, 0.4009941517128973, 0.4916372025747454, 1.3757540189251316, 0.03379720007083052], [-1.125986238782398, -1.6289501821208572, 0.9398696805867421, -0.08170029685954695, 0.9704823431161016, -1.1262790929342048], [-1.7127347694518145, -0.5025983893825684, 4.186862685341445, 0.33153155053165523, -0.04348868534119801, 0.2789790605034791], [-1.4374646943897464, 2.846154431362467, -1.8552195704954455, -0.5489957099722047, -0.27645156440470436, -2.021964278759111], [-0.8656558424202634, -1.3166553382342943, 0.43744495448576587, -0.7076346708170212, 1.1612867395083066, 1.503399391681574], [-3.090285164174917, 0.8402332201304372, 0.16157364917952333, -0.9282910042174405, 1.4842799933005888, 0.3463441850792637], [-1.4159000971090467, -2.2542491526249697, -0.15429001798460446, 0.8650887813994811, -1.2564550442410598, 3.042979099709489], [-2.9379508432863326, 0.5271922010635066, -0.9778274564987297, 1.885326951885986, 0.9277181924953881, 0.5170799132175911]], "dtype": "float64"}], ["w", {"__nd__": [[-0.22479430225523417, -0.31818464242238514, 0.4940571709196207, -0.281836567056317, -0.08771091796396684, -0.41447281747298304], [-0.10840313711851654, -0.949084591077159, 0.8481302713001143, 0.5138021538663902, -0.5609783452170521, -0.5228528552520045], [-0.04043521231004415, 0.18141318275100182, 0.16391629238507718, -0.5775002961798578, 0.07351410501798801, -0.10958520760463973], [-0.3349790856806678, 0.43751450628167593, 0.11295516566869412, 0.20646320096872042, 0.07331743696305063, -0.0858701217391654], [0.8145086071407418, -0.16612565532493725, -0.7750216706817373, -0.13380523139704187, -0.6198456213430292, 0.388844241100845], [0.5501988929908236, -0.2174777203525987, 0.1868870723787177, -0.169310992410892, 0.48256401385535624, 0.5685414791251505]], "dtype": "float64"}], ["out", {"__nd__": [[0.4123647974967628, -0.20235396583738774, -1.288204253721664, -3.0621264334266556, -1.1920402087475443, -0.37622445831259294, -0.3072824127150172, 1.0250759131539668, 0.7235398743589224, 0.2858883827661679, 1.8569733920922196, -2.2115386573372753], [1.241675389263351, -1.4800460725747926, 0.6687524513189879, 1.7989402179588199, 2.4424575191236864, -0.20093574667647884, -0.5203638179199983, 1.52596251285114, 1.1130769594342271, 0.733394313007588, -0.5842908779894079, -1.6652246110991213], [-1.8421554072085256, 4.093915950819894, -0.7813947151801371, 0.31362116686347324, 2.1175002781065464, -1.6277989699572006, -0.9569256933793637, 1.6631606237822978, 0.816622720149075, -2.1839306244840584, -1.1103186305160557, -0.41773840599185225], [-0.40737478749606126, 1.0304295905459568, -0.7504173407045005, -1.7291235775427984, -0.42417940754941774, 2.340261805651692, -1.2827056119124836, -0.09512222668310635, 0.7225215726426936, -0.26456250529251546, 2.739491436469211, -2.130169119844323], [-0.47354769829330706, 1.0368285556173593, 0.9853573528604106, 1.5497964442817542, -0.41733600178977115, 1.1987711461970136, -1.5502672232637482, -0.9989486757096833, 1.1384448558489404, -0.8482779857984768, 1.6248064027583222, 0.14426450537452923], [0.353243690743899, -0.6781941733888346, -3.334750604801741, 2.73280859852259, -1.6684115475276389, 1.1421428898258035, 0.38788334399259716, -0.6937587646209356, -1.001211682626923, 1.0737567866818187, 0.6903915040155135, -0.5224043751124285]], "dtype": "float64"}]]}, {"__nd__": [[11, 10, 11, 7, 1], [11, 10, 10, 3, 0], [4, 2, 6, 4, 8], [1, 7, 7, 4, 2], [7, 3, 0, 8, 2], [10, 11, 0, 1, 5]], "dtype": "int64"}, 12, 11, 0], "e": {"__tup__": [{"__nd__": [[4, 1, 3, 2, 10, 5, 3, 6, 1, 3, 11, 0], [5, 1, 3, 2, 10, 5, 3, 6, 1, 3, 11, 0], [10, 9, 3, 6, 1, 3, 11, 0, 0, 0, 0, 0], [10, 1, 3, 6, 1, 3, 11, 0, 0, 0, 0, 0], [10, 1, 3, 6, 1, 3, 11, 0, 0, 0, 0, 0], [1, 3, 2, 10, 5, 3, 6, 1, 3, 11, 0, 0]], "dtype": "int32"}, {"__nd__": [11, 11, 7, 7, 7, 10], "dtype": "int32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(greedy_decode, json.loads(_TESTS), 'jax', *[1e-06, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")